In [2]:
import gurobipy as gp;
from gurobipy import GRB;
import numpy as np;

In [3]:
xi = [1,2,4];

numScens = len(xi);

# Define master problem
m = gp.Model("testLP");
m.modelSense = GRB.MINIMIZE; 

x = m.addVar(vtype=GRB.CONTINUOUS, lb = 0, ub = 10, obj=0);
theta = m.addVar(vtype=GRB.CONTINUOUS, lb = 0, ub = GRB.INFINITY, obj=1);

m.setParam("OutputFlag", 0); # disable output information unless you want to take a look at the solution process

Set parameter Username
Academic license - for non-commercial use only - expires 2025-07-29


In [4]:
# Just give an arbirtrary xval initially, will change iteratively anyway
xval = 0;

# Define subproblem
sub = gp.Model("subprob");
sub.modelSense = GRB.MINIMIZE; 

y1 = sub.addVar(vtype=GRB.CONTINUOUS, lb = 0, ub = GRB.INFINITY, obj=1);
y2 = sub.addVar(vtype=GRB.CONTINUOUS, lb = 0, ub = GRB.INFINITY, obj=1);
 
# Give a reference to each constraint since we will use the dual multiplier associated with each constraint
constr = sub.addConstr(y1 - y2 == 0);
# Just give an arbitrary rhs for now, will change iteratively anyway

sub.setParam("OutputFlag", 0); # disable output information unless you want to take a look at the solution process

In [5]:
LB = 0;
UB = 1e10;
iter = 0;

while (UB-LB)*1.0/(max(abs(UB),1e-8)) > 1e-5:
  iter = iter + 1;
  # First solve the master problem, and get x value
  m.optimize();
  xval = x.x;
  thetaval = theta.x;
  LB = m.objVal;

  duals = [0]*numScens;
  tempUB = 0;
  for k in range(numScens):  
    constr.setAttr(GRB.Attr.RHS, xi[k]-xval);
    # This is a simple case, since we have complete recourse: so there is no feasibility cuts
    sub.optimize();
    tempUB = tempUB + sub.objVal*1.0/numScens;
    duals[k] = constr.pi;

  if tempUB < UB:
    UB = tempUB;
  if (UB-LB)*1.0/(max(abs(UB),1e-8)) < 1e-5:
    break;
  else:
    # If not converge, MUST be able to generate a cut -- no need to check cut violation
    # Now construct cut:
    # Cut should look like: theta >= sum_k p_k(xi[k]-x)*dual[k]
    m.addConstr(theta + (sum(duals)*1.0/numScens)*x >= np.dot(duals,xi)*1.0/numScens)
    print('cut[%d]: cutcoef = %f, cutrhs = %f' % (iter, (sum(duals)*1.0/numScens), (np.dot(duals,xi)*1.0/numScens)));
    
print("LB = ", LB);
print("UB = ", UB);
print("xval = ", xval);

cut[1]: cutcoef = 1.000000, cutrhs = 2.333333
cut[2]: cutcoef = -0.333333, cutrhs = 0.333333
cut[3]: cutcoef = 0.333333, cutrhs = 1.666667
LB =  1.0
UB =  1.0
xval =  2.0000000000000004
